In [7]:
import os
import sys
import numpy as np
import traceback
from datetime import datetime
import psycopg2
from psycopg2.extras import Json
from scipy.fft import dct, idct

# 프로젝트 루트 경로 추가
sys.path.append('../')
from core.config import ConfigLoader

# DB 연결 정보
db_cfg = ConfigLoader(config_path="../core/config.yaml").db
print("[DB CONFIG]", db_cfg)

conn = psycopg2.connect(
    host=db_cfg['host'], 
    port=db_cfg['port'], 
    user=db_cfg['user'], 
    password=db_cfg['password'], 
    dbname=db_cfg['dbname']
)
conn.autocommit = True
cur = conn.cursor()
print("DB 연결 완료")


[DB CONFIG] {'host': 'localhost', 'port': 5432, 'user': 'postgres', 'password': 'postgres', 'dbname': 'postgres'}
DB 연결 완료


In [8]:
# dct_vector 테이블 생성
create_table_sql = '''
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS dct_vector (
    id serial PRIMARY KEY,
    origin_vector_id integer NOT NULL REFERENCES origin_vector(id),
    keep_dim integer NOT NULL,
    mode varchar(16) NOT NULL DEFAULT 'low',
    original_dim integer NOT NULL,
    compressed_dim integer NOT NULL,
    compression_ratio float,
    parameters json,
    embedding vector,
    created_at timestamp,
    log text
);

-- 인덱스 생성
CREATE INDEX IF NOT EXISTS idx_dct_vector_origin_id ON dct_vector(origin_vector_id);
CREATE INDEX IF NOT EXISTS idx_dct_vector_keep_dim ON dct_vector(keep_dim);
CREATE INDEX IF NOT EXISTS idx_dct_vector_mode ON dct_vector(mode);
'''

try:
    cur.execute(create_table_sql)
    print('dct_vector 테이블 및 인덱스 생성 완료')
except Exception as e:
    print(f'테이블 생성 에러: {e}')


dct_vector 테이블 및 인덱스 생성 완료


In [9]:
# DCT 변환 설정 (routes/extract_router.py와 동일)
# routes/extract_router.py에서 사용한 keep_sizes
dct_keep_dims = [8, 16, 32, 64, 128, 256]
modes = ['low', 'high']

print(f"DCT keep dimensions: {dct_keep_dims}")
print(f"DCT modes: {modes}")
print(f"총 조합 수: {len(dct_keep_dims)} x {len(modes)} = {len(dct_keep_dims) * len(modes)}")

# 512차원 벡터에 대한 DCT 적용 확인
test_vector = np.random.randn(512)
test_dct = dct(test_vector, norm='ortho')
print(f"\\n512차원 벡터 DCT 변환 테스트:")
print(f"  원본 차원: {len(test_vector)}")
print(f"  DCT 후 차원: {len(test_dct)}")
print(f"  최대 keep_dim({max(dct_keep_dims)})이 벡터 길이보다 작음: {max(dct_keep_dims) < len(test_vector)}")


DCT keep dimensions: [8, 16, 32, 64, 128, 256]
DCT modes: ['low', 'high']
총 조합 수: 6 x 2 = 12
\n512차원 벡터 DCT 변환 테스트:
  원본 차원: 512
  DCT 후 차원: 512
  최대 keep_dim(256)이 벡터 길이보다 작음: True


In [10]:
# DCT 변환 함수 정의
def dct_transform_extract(vector, keep_dim, mode='low'):
    """
    1차원 벡터에서 DCT 변환 후 저주파/고주파 성분 추출
    core/pipeline/transformers/dct.py의 DCTTransformer와 유사한 방식
    """
    try:
        # DCT 변환 (orthogonal normalization)
        dct_coeffs = dct(vector, norm='ortho')
        
        original_dim = len(vector)
        
        if mode == 'low':
            # 저주파 성분만 유지 (처음 keep_dim개만 유지, 나머지는 0)
            dct_coeffs_filtered = dct_coeffs.copy()
            dct_coeffs_filtered[keep_dim:] = 0
            # 압축된 벡터는 처음 keep_dim개의 계수만 저장
            compressed_vector = dct_coeffs_filtered[:keep_dim]
            compressed_dim = keep_dim
            
        elif mode == 'high':
            # 고주파 성분만 유지 (처음 keep_dim개를 0으로, 나머지 유지)
            dct_coeffs_filtered = dct_coeffs.copy()
            dct_coeffs_filtered[:keep_dim] = 0
            # 압축된 벡터는 keep_dim 이후의 계수들
            compressed_vector = dct_coeffs_filtered[keep_dim:]
            compressed_dim = len(compressed_vector)
            
        else:
            raise ValueError("mode는 'low' 또는 'high'만 허용")
        
        compression_ratio = compressed_dim / original_dim
        log_msg = f"SUCCESS: keep_dim={keep_dim}, mode={mode}, {original_dim}->{compressed_dim} (ratio={compression_ratio:.3f})"
        
        return compressed_vector, original_dim, compressed_dim, compression_ratio, log_msg
        
    except Exception as e:
        error_msg = f"ERROR: keep_dim={keep_dim}, mode={mode}, error={str(e)}"
        return None, len(vector), 0, 0.0, error_msg

# 테스트
for mode in ['low', 'high']:
    test_result = dct_transform_extract(test_vector, 64, mode)
    print(f"테스트 결과 ({mode}):", test_result[4])  # log 메시지만 출력


테스트 결과 (low): SUCCESS: keep_dim=64, mode=low, 512->64 (ratio=0.125)
테스트 결과 (high): SUCCESS: keep_dim=64, mode=high, 512->448 (ratio=0.875)


In [11]:
# origin_vector에서 데이터 로드
cur.execute("""
    SELECT id, embedding, image_path, label
    FROM origin_vector 
    WHERE log LIKE 'Face detected%'  -- 얼굴이 정상 검출된 경우만
    ORDER BY id
""")

origin_vectors = cur.fetchall()
print(f"처리할 origin_vector 수: {len(origin_vectors)}")

if len(origin_vectors) > 0:
    print("샘플 데이터:")
    sample = origin_vectors[0]
    print(f"  ID: {sample[0]}")
    print(f"  이미지 경로: {sample[2]}")
    print(f"  라벨: {sample[3]}")
    print(f"  임베딩 차원: {len(sample[1])}")
    
    # 예상 총 작업량 계산
    total_combinations = len(dct_keep_dims) * len(modes)
    total_tasks = len(origin_vectors) * total_combinations
    print(f"\\n예상 총 작업량:")
    print(f"  origin_vectors: {len(origin_vectors)}")
    print(f"  keep_dims × modes: {len(dct_keep_dims)} × {len(modes)} = {total_combinations}")
    print(f"  총 DCT 변환 작업: {total_tasks}")
else:
    print("처리할 데이터가 없습니다!")


처리할 origin_vector 수: 13195
샘플 데이터:
  ID: 1
  이미지 경로: Aaron_Eckhart\Aaron_Eckhart_0001.jpg
  라벨: Aaron_Eckhart
  임베딩 차원: 5572
\n예상 총 작업량:
  origin_vectors: 13195
  keep_dims × modes: 6 × 2 = 12
  총 DCT 변환 작업: 158340


In [12]:
# 모든 keep_dim과 mode 조합으로 DCT 변환 및 저장
import json

processed_count = 0
error_count = 0
skipped_count = 0

print("DCT 변환 및 저장 시작...")

for idx, (origin_id, embedding_list, image_path, label) in enumerate(origin_vectors):
    try:
        # 임베딩 데이터 파싱 (문자열이나 다른 형태일 수 있음)
        if isinstance(embedding_list, str):
            # 문자열인 경우 JSON으로 파싱
            if embedding_list.startswith('[') and embedding_list.endswith(']'):
                embedding_data = json.loads(embedding_list)
            else:
                print(f"예상치 못한 문자열 형태 [ID:{origin_id}]: {embedding_list[:100]}...")
                continue
        elif hasattr(embedding_list, '__iter__'):
            # 이미 리스트나 배열 형태인 경우
            embedding_data = list(embedding_list)
        else:
            print(f"알 수 없는 데이터 타입 [ID:{origin_id}]: {type(embedding_list)}")
            continue
            
        # numpy 배열로 변환
        embedding_vector = np.array(embedding_data, dtype=np.float32)
        
        if len(embedding_vector) == 0:
            print(f"빈 임베딩 벡터 [ID:{origin_id}]")
            continue
            
    except Exception as parse_error:
        print(f"임베딩 파싱 에러 [ID:{origin_id}]: {str(parse_error)}")
        continue
    
    # 모든 keep_dim에 대해 처리
    for keep_dim in dct_keep_dims:
        # 각 mode에 대해 처리
        for mode in modes:
            try:
                # 중복 방지: 이미 처리된 조합인지 확인
                cur.execute("""
                    SELECT id FROM dct_vector 
                    WHERE origin_vector_id=%s AND keep_dim=%s AND mode=%s
                """, (origin_id, keep_dim, mode))
                
                if cur.fetchone():
                    skipped_count += 1
                    continue
                
                # DCT 변환 수행
                compressed_vec, original_dim, compressed_dim, compression_ratio, log_msg = dct_transform_extract(
                    embedding_vector, keep_dim, mode
                )
                
                if compressed_vec is not None:
                    # DB 저장용 파라미터
                    parameters = {
                        'keep_dim': keep_dim,
                        'mode': mode,
                        'norm': 'ortho'
                    }
                    
                    # DB 저장
                    cur.execute("""
                        INSERT INTO dct_vector 
                        (origin_vector_id, keep_dim, mode, original_dim, compressed_dim, 
                         compression_ratio, parameters, embedding, created_at, log) 
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """, (
                        origin_id, keep_dim, mode, original_dim, compressed_dim,
                        compression_ratio, Json(parameters), compressed_vec.tolist(), 
                        datetime.now(), log_msg
                    ))
                    
                    processed_count += 1
                else:
                    # 에러 발생 시 로그만 저장 (embedding은 빈 배열)
                    cur.execute("""
                        INSERT INTO dct_vector 
                        (origin_vector_id, keep_dim, mode, original_dim, compressed_dim, 
                         compression_ratio, parameters, embedding, created_at, log) 
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """, (
                        origin_id, keep_dim, mode, original_dim, 0,
                        0.0, Json({'keep_dim': keep_dim, 'mode': mode}), [], 
                        datetime.now(), log_msg
                    ))
                    error_count += 1
                
            except Exception as e:
                error_count += 1
                print(f"에러 [ID:{origin_id}, keep_dim:{keep_dim}, mode:{mode}]: {str(e)}")
                
                # 에러 로그 저장
                try:
                    cur.execute("""
                        INSERT INTO dct_vector 
                        (origin_vector_id, keep_dim, mode, original_dim, compressed_dim, 
                         compression_ratio, parameters, embedding, created_at, log) 
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """, (
                        origin_id, keep_dim, mode, len(embedding_vector), 0,
                        0.0, Json({'keep_dim': keep_dim, 'mode': mode}), [], 
                        datetime.now(), f"EXCEPTION: {str(e)}"
                    ))
                except:
                    pass  # 중복 등으로 인한 DB 에러는 무시
    
    # 주기적 진행 상황 출력
    if idx % 100 == 0:
        print(f'진행률: {idx}/{len(origin_vectors)} (처리:{processed_count}, 에러:{error_count}, 스킵:{skipped_count})')

print(f'\\nDCT 변환 완료!')
print(f'- 총 origin_vectors: {len(origin_vectors)}')
print(f'- 처리 완료: {processed_count}')
print(f'- 에러: {error_count}') 
print(f'- 스킵(중복): {skipped_count}')


DCT 변환 및 저장 시작...
진행률: 0/13195 (처리:12, 에러:0, 스킵:0)
진행률: 100/13195 (처리:1212, 에러:0, 스킵:0)
진행률: 200/13195 (처리:2412, 에러:0, 스킵:0)
진행률: 300/13195 (처리:3612, 에러:0, 스킵:0)
진행률: 400/13195 (처리:4812, 에러:0, 스킵:0)
진행률: 500/13195 (처리:6012, 에러:0, 스킵:0)
진행률: 600/13195 (처리:7212, 에러:0, 스킵:0)
진행률: 700/13195 (처리:8412, 에러:0, 스킵:0)
진행률: 800/13195 (처리:9612, 에러:0, 스킵:0)
진행률: 900/13195 (처리:10812, 에러:0, 스킵:0)
진행률: 1000/13195 (처리:12012, 에러:0, 스킵:0)
진행률: 1100/13195 (처리:13212, 에러:0, 스킵:0)
진행률: 1200/13195 (처리:14412, 에러:0, 스킵:0)
진행률: 1300/13195 (처리:15612, 에러:0, 스킵:0)
진행률: 1400/13195 (처리:16812, 에러:0, 스킵:0)
진행률: 1500/13195 (처리:18012, 에러:0, 스킵:0)
진행률: 1600/13195 (처리:19212, 에러:0, 스킵:0)
진행률: 1700/13195 (처리:20412, 에러:0, 스킵:0)
진행률: 1800/13195 (처리:21612, 에러:0, 스킵:0)
진행률: 1900/13195 (처리:22812, 에러:0, 스킵:0)
진행률: 2000/13195 (처리:24012, 에러:0, 스킵:0)
진행률: 2100/13195 (처리:25212, 에러:0, 스킵:0)
진행률: 2200/13195 (처리:26412, 에러:0, 스킵:0)
진행률: 2300/13195 (처리:27612, 에러:0, 스킵:0)
진행률: 2400/13195 (처리:28812, 에러:0, 스킵:0)
진행률: 2500/13195 (처리:30012, 에러:

In [14]:
# 결과 확인 및 통계
print("=== DCT 변환 결과 확인 ===")

# 전체 변환된 레코드 수
cur.execute("SELECT COUNT(*) FROM dct_vector")
total_count = cur.fetchone()[0]
print(f"총 변환된 레코드 수: {total_count}")

# keep_dim별 통계
cur.execute("""
    SELECT keep_dim, COUNT(*) as count, 
           AVG(compression_ratio) as avg_ratio,
           MIN(compressed_dim) as min_dim,
           MAX(compressed_dim) as max_dim
    FROM dct_vector 
    WHERE log LIKE 'SUCCESS%'
    GROUP BY keep_dim 
    ORDER BY keep_dim
""")
print("\\nkeep_dim별 통계 (성공한 변환만):")
for row in cur.fetchall():
    keep_dim, count, avg_ratio, min_dim, max_dim = row
    print(f"  keep_dim={keep_dim}: {count}개, 평균압축비={avg_ratio:.3f}, 차원범위={min_dim}-{max_dim}")

# mode별 통계
cur.execute("""
    SELECT mode, COUNT(*) as count,
           AVG(compression_ratio) as avg_ratio,
           AVG(compressed_dim) as avg_dim
    FROM dct_vector 
    WHERE log LIKE 'SUCCESS%'
    GROUP BY mode 
    ORDER BY mode
""")
print("\\nmode별 통계 (성공한 변환만):")
for row in cur.fetchall():
    mode, count, avg_ratio, avg_dim = row
    print(f"  {mode}: {count}개, 평균압축비={avg_ratio:.3f}, 평균차원={avg_dim:.1f}")

# 성공/실패 통계
cur.execute("""
    SELECT 
        CASE 
            WHEN log LIKE 'SUCCESS%' THEN 'Success'
            WHEN log LIKE 'ERROR%' THEN 'Error'
            ELSE 'Exception'
        END as status,
        COUNT(*) as count
    FROM dct_vector 
    GROUP BY 
        CASE 
            WHEN log LIKE 'SUCCESS%' THEN 'Success'
            WHEN log LIKE 'ERROR%' THEN 'Error'
            ELSE 'Exception'
        END
""")
print("\\n변환 결과 통계:")
for status, count in cur.fetchall():
    print(f"  {status}: {count}개")

# 샘플 데이터 조회 (각 mode별로)
try:
    for mode in ['low', 'high']:
        cur.execute("""
            SELECT dv.keep_dim, dv.compressed_dim, dv.compression_ratio, 
                   ov.image_path, ov.label
            FROM dct_vector dv
            JOIN origin_vector ov ON dv.origin_vector_id = ov.id
            WHERE dv.mode = %s AND dv.log LIKE 'SUCCESS%'
            ORDER BY dv.keep_dim, dv.origin_vector_id
            LIMIT 3
        """, (mode,))
        
        results = cur.fetchall()
        print(f"\\n샘플 데이터 ({mode} mode):")
        
        if results:
            for row in results:
                if len(row) >= 5:
                    keep_dim, compressed_dim, ratio, image_path, label = row
                    print(f"  keep_dim={keep_dim}, 압축차원={compressed_dim}, 압축비={ratio:.3f}, {label}/{image_path}")
                else:
                    print(f"  데이터 형태가 예상과 다름: {row}")
        else:
            print(f"  {mode} mode에 대한 샘플 데이터가 없습니다.")
            
except Exception as e:
    print(f"샘플 데이터 조회 중 에러: {str(e)}")
    # 간단한 대체 쿼리로 샘플 확인
    try:
        cur.execute("SELECT keep_dim, mode, COUNT(*) FROM dct_vector WHERE log LIKE 'SUCCESS%' GROUP BY keep_dim, mode ORDER BY keep_dim, mode LIMIT 5")
        simple_results = cur.fetchall()
        print("\\n대체 샘플 데이터:")
        for row in simple_results:
            print(f"  keep_dim={row[0]}, mode={row[1]}, count={row[2]}")
    except Exception as e2:
        print(f"대체 쿼리도 실패: {str(e2)}")

print("\\n=== 작업 완료 ===")


=== DCT 변환 결과 확인 ===
총 변환된 레코드 수: 158340
\nkeep_dim별 통계 (성공한 변환만):
  keep_dim=8: 26390개, 평균압축비=0.500, 차원범위=8-504
  keep_dim=16: 26390개, 평균압축비=0.500, 차원범위=16-496
  keep_dim=32: 26390개, 평균압축비=0.500, 차원범위=32-480
  keep_dim=64: 26390개, 평균압축비=0.500, 차원범위=64-448
  keep_dim=128: 26390개, 평균압축비=0.500, 차원범위=128-384
  keep_dim=256: 26390개, 평균압축비=0.500, 차원범위=256-256
\nmode별 통계 (성공한 변환만):
  high: 79170개, 평균압축비=0.836, 평균차원=428.0
  low: 79170개, 평균압축비=0.164, 평균차원=84.0
\n변환 결과 통계:
  Success: 158340개
샘플 데이터 조회 중 에러: tuple index out of range
\n대체 샘플 데이터:
  keep_dim=8, mode=high, count=13195
  keep_dim=8, mode=low, count=13195
  keep_dim=16, mode=high, count=13195
  keep_dim=16, mode=low, count=13195
  keep_dim=32, mode=high, count=13195
\n=== 작업 완료 ===


In [15]:
# DB 연결 종료
cur.close()
conn.close()
print("DB 연결 종료")


DB 연결 종료
